# 📊 PyQt6 – Lekce 11: QTableWidget – Tabulková data

Mnoho aplikací (sklady, kontakty, výsledky testů, databáze) zobrazuje data v tabulce.
`QTableWidget` je připravený widget pro editovatelné tabulky přímo v okně.

---

## 📦 Klíčové metody QTableWidget

| Metoda | Co dělá |
|---|---|
| `QTableWidget(řádky, sloupce)` | Vytvoří tabulku s daným rozměrem |
| `.setHorizontalHeaderLabels([...])` | Nastaví záhlaví sloupců |
| `.setItem(r, s, QTableWidgetItem("text"))` | Vloží buňku na pozici (řádek, sloupec) |
| `.item(r, s).text()` | Přečte text buňky |
| `.insertRow(r)` | Přidá nový řádek |
| `.removeRow(r)` | Odebere řádek |
| `.rowCount()` | Počet řádků |
| `.columnCount()` | Počet sloupců |
| `.currentRow()` | Index aktuálně vybraného řádku |
| `.setSortingEnabled(True)` | Povolí řazení kliknutím na záhlaví |
| `.setEditTriggers(...)` | Kdy lze buňku editovat |

---

## 🟢 Ukázka 1 – Základní tabulka s pevnými daty

In [1]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QTableWidget, QTableWidgetItem
)

# Data – seznam slovníků (jako řádky z databáze)
STUDENTI = [
    {"jmeno": "Anna Nováková",   "trida": "IT4A", "prumer": "1.3"},
    {"jmeno": "Petr Svoboda",    "trida": "IT4B", "prumer": "2.1"},
    {"jmeno": "Jana Procházková","trida": "IT4A", "prumer": "1.7"},
    {"jmeno": "Tomáš Kovář",     "trida": "IT4C", "prumer": "3.0"},
]

class TabulkaStudentu(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Seznam studentů")
        self.resize(500, 200)

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        # Vytvoříme tabulku: tolik řádků, kolik máme studentů; 3 sloupce
        tabulka = QTableWidget(len(STUDENTI), 3)
        tabulka.setHorizontalHeaderLabels(["Jméno", "Třída", "Průměr"])
        tabulka.horizontalHeader().setStretchLastSection(True)  # Poslední sloupec roztáhne
        tabulka.setSortingEnabled(True)   # Klik na záhlaví = seřadit

        for radek, student in enumerate(STUDENTI):
            tabulka.setItem(radek, 0, QTableWidgetItem(student["jmeno"]))
            tabulka.setItem(radek, 1, QTableWidgetItem(student["trida"]))
            tabulka.setItem(radek, 2, QTableWidgetItem(student["prumer"]))

        layout.addWidget(tabulka)


app = QApplication.instance() or QApplication([])
okno = TabulkaStudentu()
okno.show()
app.exec()

0

---

## 🟢 Ukázka 2 – Přidávání a mazání řádků

Uživatel může přidávat nové záznamy přes formulář a mazat vybraný řádek.

In [2]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QHBoxLayout, QTableWidget, QTableWidgetItem,
    QPushButton, QLineEdit, QLabel, QAbstractItemView
)

class SpravcaKontaktu(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Správce kontaktů")
        self.resize(550, 350)

        central = QWidget()
        self.setCentralWidget(central)
        hlavni = QVBoxLayout(central)

        # --- Tabulka ---
        self.tabulka = QTableWidget(0, 2)   # 0 řádků na začátku, 2 sloupce
        self.tabulka.setHorizontalHeaderLabels(["Jméno", "Telefon"])
        self.tabulka.horizontalHeader().setStretchLastSection(True)
        self.tabulka.setSelectionBehavior(QAbstractItemView.SelectionBehavior.SelectRows)  # Vybírá celý řádek
        hlavni.addWidget(self.tabulka)

        # --- Formulář pro přidání ---
        formular = QHBoxLayout()
        self.pole_jmeno = QLineEdit()
        self.pole_jmeno.setPlaceholderText("Jméno")
        self.pole_tel = QLineEdit()
        self.pole_tel.setPlaceholderText("Telefon")
        btn_pridat = QPushButton("➕ Přidat")
        btn_smazat = QPushButton("🗑️ Smazat vybraný")

        btn_pridat.clicked.connect(self.pridat_radek)
        btn_smazat.clicked.connect(self.smazat_radek)

        formular.addWidget(QLabel("Jméno:"))
        formular.addWidget(self.pole_jmeno)
        formular.addWidget(QLabel("Tel:"))
        formular.addWidget(self.pole_tel)
        formular.addWidget(btn_pridat)
        formular.addWidget(btn_smazat)
        hlavni.addLayout(formular)

        self.statusBar().showMessage("Celkem kontaktů: 0")

    def pridat_radek(self):
        jmeno = self.pole_jmeno.text().strip()
        tel   = self.pole_tel.text().strip()
        if not jmeno:
            self.statusBar().showMessage("⚠️ Jméno nesmí být prázdné!")
            return
        r = self.tabulka.rowCount()   # Nový index = počet stávajících řádků
        self.tabulka.insertRow(r)
        self.tabulka.setItem(r, 0, QTableWidgetItem(jmeno))
        self.tabulka.setItem(r, 1, QTableWidgetItem(tel))
        self.pole_jmeno.clear()
        self.pole_tel.clear()
        self.statusBar().showMessage(f"Celkem kontaktů: {self.tabulka.rowCount()}")

    def smazat_radek(self):
        radek = self.tabulka.currentRow()
        if radek >= 0:
            self.tabulka.removeRow(radek)
            self.statusBar().showMessage(f"Celkem kontaktů: {self.tabulka.rowCount()}")
        else:
            self.statusBar().showMessage("⚠️ Nejprve vyberte řádek!")


app = QApplication.instance() or QApplication([])
okno = SpravcaKontaktu()
okno.show()
app.exec()

0

---

## 🟡 Ukázka 3 – Čtení a editace buněk, export do CSV

Procházíme celou tabulku a exportujeme data do CSV souboru.

In [3]:
import csv
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QHBoxLayout, QTableWidget, QTableWidgetItem,
    QPushButton, QLabel, QFileDialog
)

PRODUKTY = [
    ["Notebook",  "15999", "5"],
    ["Myš",       "399",   "20"],
    ["Klávesnice","799",   "15"],
    ["Monitor",   "6999",  "8"],
]

class SkladApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Skladová evidence")
        self.resize(500, 250)

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        self.tabulka = QTableWidget(len(PRODUKTY), 3)
        self.tabulka.setHorizontalHeaderLabels(["Produkt", "Cena (Kč)", "Sklad (ks)"])
        self.tabulka.horizontalHeader().setStretchLastSection(True)

        for r, radek in enumerate(PRODUKTY):
            for s, hodnota in enumerate(radek):
                self.tabulka.setItem(r, s, QTableWidgetItem(hodnota))

        layout.addWidget(self.tabulka)

        tlacitka = QHBoxLayout()
        btn_celkova = QPushButton("🧮 Spočítat celkovou hodnotu skladu")
        btn_export  = QPushButton("📤 Exportovat CSV")
        btn_celkova.clicked.connect(self.spocitej)
        btn_export.clicked.connect(self.exportuj_csv)
        tlacitka.addWidget(btn_celkova)
        tlacitka.addWidget(btn_export)
        layout.addLayout(tlacitka)

        self.statusBar().showMessage("Upravujte buňky přímo v tabulce")

    def spocitej(self):
        celkem = 0
        for r in range(self.tabulka.rowCount()):
            cena_item = self.tabulka.item(r, 1)
            sklad_item = self.tabulka.item(r, 2)
            if cena_item and sklad_item:
                try:
                    celkem += float(cena_item.text()) * float(sklad_item.text())
                except ValueError:
                    pass
        self.statusBar().showMessage(f"Celková hodnota skladu: {celkem:,.0f} Kč")

    def exportuj_csv(self):
        cesta, _ = QFileDialog.getSaveFileName(self, "Uložit jako", "sklad.csv", "CSV (*.csv)")
        if not cesta:
            return
        with open(cesta, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            # Záhlaví
            hlavicky = [self.tabulka.horizontalHeaderItem(s).text()
                        for s in range(self.tabulka.columnCount())]
            writer.writerow(hlavicky)
            # Data
            for r in range(self.tabulka.rowCount()):
                radek = []
                for s in range(self.tabulka.columnCount()):
                    item = self.tabulka.item(r, s)
                    radek.append(item.text() if item else "")
                writer.writerow(radek)
        self.statusBar().showMessage(f"Exportováno: {cesta}")


app = QApplication.instance() or QApplication([])
okno = SkladApp()
okno.show()
app.exec()

0

---

## 📋 Shrnutí lekce

| Operace | Kód |
|---|---|
| Vytvořit tabulku | `QTableWidget(řádky, sloupce)` |
| Záhlaví sloupců | `.setHorizontalHeaderLabels([...])` |
| Zapsat buňku | `.setItem(r, s, QTableWidgetItem("text"))` |
| Přečíst buňku | `.item(r, s).text()` |
| Přidat řádek | `.insertRow(.rowCount())` |
| Smazat řádek | `.removeRow(.currentRow())` |
| Zakázat editaci | `.setEditTriggers(QAbstractItemView.NoEditTriggers)` |

## ✏️ Úkoly

1. Vytvořte aplikaci **Telefonní seznam** s sloupci Jméno, Telefon, E-mail. Přidejte formulář pro přidání kontaktu a tlačítko pro mazání.
2. Přidejte do tabulky **vyhledávání** – textové pole, po jehož změně skryje řádky, které neobsahují hledaný text (hint: `.setRowHidden(r, True/False)`).
3. Přidejte tlačítko **Importovat CSV** (opak Exportu).